# 全链长，bb分析

In [1]:
import torch
import numpy as np
import plotly.graph_objects as go

def interactive_visualize(
    coords_clean: torch.Tensor,
    coords_noisy: torch.Tensor,
    atom_idx: int = 1,  # CA  # 1
    output_html: str = None
):
    """Plotly 交互式 3D 图，可旋转缩放，适合调试"""
    clean_ca = coords_clean[:, atom_idx].cpu().numpy()
    noisy_ca = coords_noisy[:, atom_idx].cpu().numpy()
    displacement = np.linalg.norm(clean_ca - noisy_ca, axis=1)
    
    fig = go.Figure()
    
    # 原始结构
    fig.add_trace(go.Scatter3d(
        x=clean_ca[:, 0], y=clean_ca[:, 1], z=clean_ca[:, 2],
        mode='markers',
        name='Clean',
        line=dict(width=2, color='blue'),
        marker=dict(size=3, color='blue', opacity=0.7)
    ))
    
    # 加噪结构（用颜色编码位移大小）
    fig.add_trace(go.Scatter3d(
        x=noisy_ca[:, 0], y=noisy_ca[:, 1], z=noisy_ca[:, 2],
        # mode='lines+markers',
        # name='Noisy',
        # marker=dict(
        #     size=5,
        #     color=displacement,
        #     colorscale='RdYlBu_r',  # 红=大扰动，蓝=小扰动
        #     colorbar=dict(title="Displacement (Å)"),
        #     opacity=0.8
        # )
        mode='markers',
        name='Noisy',
        line=dict(width=1, color='red'),
        marker=dict(size=2, color='red', opacity=0.7)
    ))
    
    fig.update_layout(
        title="Clean vs Noisy (Color = Displacement)",
        scene=dict(
            xaxis_title='X (Å)',
            yaxis_title='Y (Å)',
            zaxis_title='Z (Å)',
            aspectmode='data'  # 保持 1:1:1 比例
        ),
        width=1000,
        height=800
    )
    
    if output_html:
        fig.write_html(output_html)
        print(f"✅ Interactive plot saved: {output_html}")
    
    fig.show()
    # return fig


# save
"""

# 预测对比原始数据
torch.save({
    'perturb': inputs['cord-p'],
    'pre': pred,
    'clean': tgt
}, '/root/private_data/luog/codex/IgGM/see/seefile/perturb_per_clean.pt')


"""


# 加载文件
loaded_dict = torch.load(
    '/root/private_data/luog/codex/IgGM/see/seefile/S26_1779781000.pt',
    map_location='cpu',    
    weights_only=True       
)

# 使用


a=100
interactive_visualize(
    coords_clean = loaded_dict['clean'].squeeze(0).detach()[:,:,:] ,
    coords_noisy = loaded_dict['perturb'].squeeze(0).detach()[:,:,:] ,
)
# interactive_visualize(
#     coords_clean = loaded_dict['perturb'].squeeze(0).detach() ,
#     coords_noisy = loaded_dict['pre'].squeeze(0).detach() ,
# )

interactive_visualize(
    coords_clean = loaded_dict['clean'].squeeze(0).detach() ,
    coords_noisy = loaded_dict['pre'].squeeze(0).detach() ,
)

# a=40
# print(loaded_dict['clean'].squeeze(0).detach()[:a,:,:])
# print(loaded_dict['pre'].squeeze(0).detach()[:a,:,:])


# a=26 # 35
# print(loaded_dict['clean'].squeeze(0).detach()[a,:,:])
# print(loaded_dict['pre'].squeeze(0).detach()[a,:,:])
# interactive_visualize(
#     coords_clean = loaded_dict['clean'].squeeze(0).detach()[10:40,:,:] ,
#     coords_noisy = loaded_dict['pre'].squeeze(0).detach()[10:40,:,:] ,
# )

# 逐个氨基酸分析

In [ ]:
# import torch
# import numpy as np
# import plotly.graph_objects as go

# def interactive_visualize(
#     coords_clean: torch.Tensor,
#     coords_noisy: torch.Tensor,
#     res_idx: int = 0,  
#     output_html: str = None
# ):
#     """Plotly 交互式 3D 图，用于调试 atom14 表示法中虚拟原子的空间分布"""
    
#     # 提取坐标时加上 .copy()，避免原地修改影响外部数据
#     clean_atoms = coords_clean[res_idx].cpu().numpy().copy()
#     noisy_atoms = coords_noisy[res_idx].cpu().numpy().copy()
    
#     # 【新增：添加虚伪的微小扰动】
#     # 给虚拟原子（索引 4~13）加上正态分布的随机位移，以便在 3D 图中散开
#     jitter_scale = 0.1  # 扰动大小 (Å)，如果还是看不清可以稍微调大
#     clean_atoms[4:14] += np.random.normal(scale=jitter_scale, size=(10, 3))
#     noisy_atoms[4:14] += np.random.normal(scale=jitter_scale, size=(10, 3))
    
#     # 为14个原子生成悬停标签，方便你在网页上确认是哪个原子
#     atom_labels = []
#     for i in range(14):
#         if i == 0: atom_labels.append("0: N")
#         elif i == 1: atom_labels.append("1: CA")
#         elif i == 2: atom_labels.append("2: C")
#         elif i == 3: atom_labels.append("3: O")
#         else: atom_labels.append(f"{i}")
#         # else: atom_labels.append(f"{i}: Virtual/Side")

    
#     atom_labels2 = []
#     for i in range(14):
#         if i == 0: atom_labels2.append("0: -N")
#         elif i == 1: atom_labels2.append("1: -CA")
#         elif i == 2: atom_labels2.append("2: -C")
#         elif i == 3: atom_labels2.append("3: -O")
#         else: atom_labels2.append(f"{i}-")

#     # 颜色编码：N设为绿色，O设为橙色，CA/C设为灰色，虚拟原子设为红色
#     marker_colors = ['green' if i==0 else 'orange' if i==3 else 'gray' if i in [1,2] else 'red' for i in range(14)]

#     fig = go.Figure()
    
#     # 原始/真实结构 (Clean) 
#     fig.add_trace(go.Scatter3d(
#         # x=clean_atoms[5:, 0], y=clean_atoms[5:, 1], z=clean_atoms[5:, 2],
#         # x=clean_atoms[:4, 0], y=clean_atoms[:4, 1], z=clean_atoms[:4, 2],
#         x=clean_atoms[:, 0], y=clean_atoms[:, 1], z=clean_atoms[:, 2],
#         mode='markers+text',
#         text=atom_labels,
#         textposition="bottom center",
#         name='Clean (Target)',
#         marker=dict(size=6, color=marker_colors, symbol='circle-open', opacity=0.8) # 'blue'
#     ))
    
#     # 预测结构 (Predicted) 
#     fig.add_trace(go.Scatter3d(
#         # x=noisy_atoms[5:, 0], y=noisy_atoms[5:, 1], z=noisy_atoms[5:, 2],
#         # x=noisy_atoms[:4, 0], y=noisy_atoms[:4, 1], z=noisy_atoms[:4, 2],
#         x=noisy_atoms[:, 0], y=noisy_atoms[:, 1], z=noisy_atoms[:, 2],
#         mode='markers+text',
#         text=atom_labels2,
#         textposition="top center",
#         name='Predicted',
#         marker=dict(size=5, color=marker_colors, opacity=0.9)
#     ))
    
#     fig.update_layout(
#         title=f"Residue {res_idx} Debug: Green=N, Orange=O, Red=Virtual Atoms (with Jitter)",
#         scene=dict(
#             xaxis_title='X (Å)',
#             yaxis_title='Y (Å)',
#             zaxis_title='Z (Å)',
#             aspectmode='data'  
#         ),
#         width=1000,
#         height=800
#     )
    
#     if output_html:
#         fig.write_html(output_html)
#         print(f"✅ Interactive plot saved: {output_html}")
    
#     fig.show()
    

# # 加载文件
# loaded_dict = torch.load(
#     '/root/private_data/luog/codex/IgGM/see/seefile/valB_1778421415.pt', # val_1778319441.pt.pt////val_1778336764.pt.pt
#     map_location='cpu',    
#     weights_only=True       
# )
# # GNIS: val_1778386818.pt
# # cdrmse：valA_1778346400.pt

# # 使用示例：检查第 0 个残基的预测情况（你可以修改 res_idx 遍历不同残基）
# interactive_visualize(
    
#     # loaded_dict['perturb'].squeeze(0).detach()[20:40,:,:] ,
#     # loaded_dict['pre'].squeeze(0).detach()[20:40,:,:] ,
    
#     coords_clean = loaded_dict['clean'].squeeze(0).detach()[213:,:,:] ,
#     coords_noisy = loaded_dict['pre'].squeeze(0).detach()[213:,:,:] ,
#     # coords_noisy = loaded_dict['pre'].squeeze(0).detach()[20:40,:,:] ,
#     res_idx = 25  # 传入残基索引
# )

In [ ]:
# # loaded_dict['clean'].squeeze(0).detach().shape
# # # a ="QDQLQQSGAELVRPGASVKLSCKALGYIFTDYEIHWVKQTPVHGLEWIGGIHPGSSGTAYNQKFKGKATLTADKSSTTAFMELSSLTSEDSAVYYCTRKDYWGQGTLVTVSAAKTTAPSVYPLVPVCGGTTGSSVTLGCLVKGYFPEPVTLTWNSGSLSSGVHTFPALLQSGLYTLSSSVTVTSNTWPSQTITCNVAHPASSTKVDKKIEPRV"
# # # a[25]
# # H = "QDQLQQSGAELVRPGASVKLSCKALGYIFTDYEIHWVKQTPVHGLEWIGGIHPGSSGTAYNQKFKGKATLTADKSSTTAFMELSSLTSEDSAVYYCTRKDYWGQGTLVTVSAAKTTAPSVYPLVPVCGGTTGSSVTLGCLVKGYFPEPVTLTWNSGSLSSGVHTFPALLQSGLYTLSSSVTVTSNTWPSQTITCNVAHPASSTKVDKKIEPRV"
# # L="DIKMTQSPSSMYTSLGERVTITCKASQDINSFLTWFLQKPGKSPKTLIYRANRLMIGVPSRFSGSGSGQTYSLTISSLEYEDMGIYYCLQYDDFPLTFGAGTKLDLKRADAAPTVSIFPPSSEQLTSGTASVVCFLNNFYPKEINVKWKIDGSERQNGVLDSWTEQDSKDSTYSMSSTLTLTKDEYERHNSYTCEATHKTSTSPIVKSFNRNEC"
# # len(H),len(L)

# for i in range(0, 100):
#     if loaded_dict['clean'].squeeze(0).detach()[213+i,-1,-1]+183 >20:
#         print(i)

In [ ]:
import torch
import numpy as np
import plotly.graph_objects as go

# 【新增函数：计算并打印虚拟原子到 N 和 O 的距离】
def print_virtual_atom_distances(coords_noisy: torch.Tensor, res_idx: int):
    """计算预测结构中虚拟原子(4-13)到 N(0) 和 O(3) 的欧氏距离(埃)"""
    # 获取特定残基的坐标 (注意：这里直接计算原始预测坐标，不加 jitter)
    atoms = coords_noisy[res_idx].cpu().numpy()
    
    n_coord = atoms[0]
    o_coord = atoms[3]
    
    print(f"\n=== 残基 {res_idx} 预测结构(Noisy) 虚拟原子距离分析 ===")
    print(f"{'Atom':<8} | {'Dist to N (Å)':<15} | {'Dist to O (Å)':<15}")
    print("-" * 42)
    
    for i in range(4, 14):
        v_coord = atoms[i]
        dist_to_n = np.linalg.norm(v_coord - n_coord)
        dist_to_o = np.linalg.norm(v_coord - o_coord)
        print(f"Atom {i:<3} | {dist_to_n:<15.4f} | {dist_to_o:<15.4f}")
    print("==========================================\n")


def interactive_visualize(
    coords_clean: torch.Tensor,
    coords_noisy: torch.Tensor,
    res_idx: int = 0,  
    output_html: str = None
):
    """Plotly 交互式 3D 图，用于调试 atom14 表示法中虚拟原子的空间分布"""
    
    # 提取坐标时加上 .copy()，避免原地修改影响外部数据
    clean_atoms = coords_clean[res_idx].cpu().numpy().copy()
    noisy_atoms = coords_noisy[res_idx].cpu().numpy().copy()
    
    # 【新增：添加虚伪的微小扰动】
    # 给虚拟原子（索引 4~13）加上正态分布的随机位移，以便在 3D 图中散开
    jitter_scale = 0.1  # 扰动大小 (Å)，如果还是看不清可以稍微调大
    clean_atoms[4:14] += np.random.normal(scale=jitter_scale, size=(10, 3))
    noisy_atoms[4:14] += np.random.normal(scale=jitter_scale, size=(10, 3))
    
    # 为14个原子生成悬停标签，方便你在网页上确认是哪个原子
    atom_labels = []
    for i in range(14):
        if i == 0: atom_labels.append("0: N")
        elif i == 1: atom_labels.append("1: CA")
        elif i == 2: atom_labels.append("2: C")
        elif i == 3: atom_labels.append("3: O")
        else: atom_labels.append(f"{i}")
        # else: atom_labels.append(f"{i}: Virtual/Side")

    
    atom_labels2 = []
    for i in range(14):
        if i == 0: atom_labels2.append("0: -N")
        elif i == 1: atom_labels2.append("1: -CA")
        elif i == 2: atom_labels2.append("2: -C")
        elif i == 3: atom_labels2.append("3: -O")
        else: atom_labels2.append(f"{i}-")

    # 颜色编码：N设为绿色，O设为橙色，CA/C设为灰色，虚拟原子设为红色
    marker_colors = ['green' if i==0 else 'orange' if i==3 else 'gray' if i in [1,2] else 'red' for i in range(14)]

    fig = go.Figure()
    
    # 原始/真实结构 (Clean) 
    fig.add_trace(go.Scatter3d(
        x=clean_atoms[:, 0], y=clean_atoms[:, 1], z=clean_atoms[:, 2],
        mode='markers+text',
        text=atom_labels,
        textposition="bottom center",
        name='Clean (Target)',
        marker=dict(size=6, color=marker_colors, symbol='circle-open', opacity=0.8) # 'blue'
    ))
    
    # 预测结构 (Predicted) 
    fig.add_trace(go.Scatter3d(
        x=noisy_atoms[:, 0], y=noisy_atoms[:, 1], z=noisy_atoms[:, 2],
        mode='markers+text',
        text=atom_labels2,
        textposition="top center",
        name='Predicted',
        marker=dict(size=5, color=marker_colors, opacity=0.9)
    ))
    
    fig.update_layout(
        title=f"Residue {res_idx} Debug: Green=N, Orange=O, Red=Virtual Atoms (with Jitter)",
        scene=dict(
            xaxis_title='X (Å)',
            yaxis_title='Y (Å)',
            zaxis_title='Z (Å)',
            aspectmode='data'  
        ),
        width=1000,
        height=800
    )
    
    if output_html:
        fig.write_html(output_html)
        print(f"✅ Interactive plot saved: {output_html}")
    
    fig.show()
    

# 加载文件
loaded_dict = torch.load(
    '/root/private_data/luog/codex/IgGM/see/seefile/S26_1779767747.pt', 
    map_location='cpu',    
    weights_only=True       
)

# 提取你想要查看的数据片段
coords_clean_segment = loaded_dict['clean'].squeeze(0).detach()[:,:,:] # clean
coords_noisy_segment = loaded_dict['pre'].squeeze(0).detach()[:,:,:] # pre 
target_res_idx = 30

# 【新增调用：在画图前打印距离信息】
print_virtual_atom_distances(
    coords_noisy=coords_noisy_segment, 
    res_idx=target_res_idx
)

# 使用示例：检查残基的预测情况
interactive_visualize(
    coords_clean = coords_clean_segment,
    coords_noisy = coords_noisy_segment,
    res_idx = target_res_idx  
)


=== 残基 30 预测结构(Noisy) 虚拟原子距离分析 ===
Atom     | Dist to N (Å)   | Dist to O (Å)  
------------------------------------------
Atom 4   | 3.6016          | 2.3203         
Atom 5   | 3.9102          | 2.8247         
Atom 6   | 4.3408          | 3.3319         
Atom 7   | 3.9328          | 2.9718         
Atom 8   | 3.8917          | 2.9214         
Atom 9   | 3.5703          | 2.3132         
Atom 10  | 4.1481          | 2.9040         
Atom 11  | 4.3811          | 3.1481         
Atom 12  | 3.5544          | 2.2836         
Atom 13  | 3.5479          | 2.2374         

